In [ ]:
# Load packages
import pandas as pd
import seaborn as sns
from ydata_profiling import ProfileReport

In [4]:
df = pd.read_csv("HotelBookings.csv")

In [ ]:
# defining categorical variables
categorical_cols = ['hotel','meal','country','market_segment','distribution_channel',
                    'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent'
                    , 'company', 'customer_type']

# convert chosen variables to categorical type
df[categorical_cols] = df[categorical_cols].astype("category")

In [ ]:
# create new datetime column by concatenating day, month, year columns
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str)
)

# drop redundant columns
df = df.drop(columns=[
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_day_of_month'
])

In [ ]:
# create new binary column on whether reserved and assigned room type are the same
df['room_changed'] = (
    df['reserved_room_type'].astype(str)
    != df['assigned_room_type'].astype(str)
).astype(int)

In [ ]:
# fill in missing rows in children column with zero
df['children'] = df['children'].fillna(0)

In [ ]:
# create a new binary column on whether an agent is booked
df['agent_booked'] = df['agent'].notna().astype(int)

In [ ]:
# tackling rows with zero values in adr column
df[df['adr']==0]

In [ ]:
# adr is 0, doesnt mean booking is cancelled (check)
pd.crosstab(df['is_canceled'], df['adr'].eq(0))

In [ ]:
# 0 values for adr, stays_in_weekend_nights
# and stays_in_week_nights seem suspicious
# flag out the rows

df['zero_night_zero_adr'] = (
    (df['stays_in_weekend_nights'] == 0) &
    (df['stays_in_week_nights'] == 0) &
    (df['adr'] == 0)
)

df['zero_night_zero_adr'].value_counts()
# 517 such rows

In [ ]:
# check no.of guest for possibly problematic rows
df[
    (df['adr'] == 0) &
    (df['stays_in_weekend_nights'] == 0) &
    (df['stays_in_week_nights'] == 0)
][['adults', 'children', 'babies']].value_counts()

# have 0 adults, children and babies seems off

In [ ]:
#investigate records
zero_records = df[
    (df['stays_in_weekend_nights'] == 0) &
    (df['stays_in_week_nights'] == 0) &
    (df['adr'] == 0)
]

pd.crosstab(
    zero_records['adults'],
    zero_records['is_canceled']
)
# There are 46 zero-night/zero-ADR records with no adults, 
# and those may be more suspicious than the 471 records that have 1-3 adults.

In [ ]:
# remove records with 0 adults, 0 children, 0 babies, 0 nights, and 0
# ADR were removed because they contain no recorded guests, stay, or accommodation revenue.
df.shape
df = df[
    ~(
        (df['adults'] == 0) &
        (df['children'] == 0) &
        (df['babies'] == 0) &
        (df['stays_in_weekend_nights'] == 0) &
        (df['stays_in_week_nights'] == 0) &
        (df['adr'] == 0)
    )
]

df.shape

# 46 rows removed

In [ ]:
# from profile report, extreme adr value of 5400, likely misinput
# replace extreme value with median

 median_adr = df['adr'].median()

df.loc[df['adr'] == 5400, 'adr'] = median_adr

In [ ]:
# profile report some value for adults seem higher than most of the other rows
#, potential outliers
df[df['adults'] >= 10][[
    'hotel',
    'adults',
    'children',
    'babies',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adr',
    'is_canceled'
]].sort_values('adults', ascending=False)

# would not remove rows solely because adults is high.
# is_canceled is target variable, these records are actually potentially important
# because they contain information about canceled bookings, all of them are recorded as cancelled bookings
# non-zero no. of stays as well

In [ ]:
# Check rows where country is not stated
df[df['country'].isna()]

# missing country information itself may be meaningful, place them all under new category 'Unknown'
df['country'] = df['country'].fillna('Unknown')

In [ ]:
# found a two letter country code
# CHN → ISO alpha-3
# CN  → ISO alpha-2
# both are china
# check for other two-letter codes
df[df['country'].str.len() != 3]['country'].value_counts()

In [ ]:
# converting CN to CHN for consistency
df['country'] = df['country'].replace({
    'CN': 'CHN'
})

In [ ]:
df['meal'].unique()
df['meal'].value_counts() 
# 'Undefined' recorded for meal in some rows
# 'Undefined' probably means no meal package
# so we just place under SC(no meal package) for now
df['meal'] = df['meal'].replace({'Undefined':'SC'})
# after changing all the 'undefined' to 'SC', check value count
df['meal'].value_counts()

In [ ]:
# Generate a simple profiling report, display, and save to HTML file
profile = ProfileReport(df)

profile.to_notebook_iframe()
profile.to_file("hotelbookings_profiling_report.html")